# 📦 FRAMEWORK COMPLETO DE PREPROCESAMIENTO
Guía metodológica + código Python listo para usar

## 📋 FASE 0: COMPRENSIÓN DEL PROBLEMA

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
df = pd.read_csv('data.csv')
df.head(), df.info(), df.describe()

In [ ]:
print('Filas, columnas:', df.shape)
print('Memoria:', df.memory_usage(deep=True).sum()/1024**2, 'MB')
df.dtypes.value_counts()

In [ ]:
df['target'].value_counts(normalize=True).plot.bar(title='Distribución target');

## 📊 FASE 1: ANÁLISIS EXPLORATORIO

In [ ]:
# Duplicados
print('Duplicados:', df.duplicated().sum())
# Missing
import missingno as msno
msno.matrix(df)

In [ ]:
# Outliers IQR
def outliers_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - k*iqr, q3 + k*iqr
    return serie[(serie<lim_inf)|(serie>lim_sup)]
outliers_iqr(df['x']).head()

In [ ]:
# Outliers Isolation Forest
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.01).fit_predict(df[['x']])
df_iso = df[iso==-1]
print('Outliers detectados:', len(df_iso))

In [ ]:
# Univariado numérico
sns.histplot(df['x'], kde=True)
plt.figure()
sns.boxplot(y=df['x'])

In [ ]:
# Univariado categórico
vc = df['cat'].value_counts(normalize=True)
vc.head(10).plot.bar(title='Top categorías')
print('Cardinalidad:', df['cat'].nunique())
print('Raras (<5%):', vc[vc<0.05].index.tolist())

In [ ]:
# Correlación
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=False, cmap='coolwarm')

## 🧹 FASE 2: LIMPIEZA BÁSICA

In [ ]:
# Strings
df['cat'] = df['cat'].str.strip().str.lower()
# Fechas
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
# Rangos imposibles
df = df[(df['edad']>=0)&(df['edad']<=120)]
# Duplicados
df.drop_duplicates(inplace=True)

## 🔧 FASE 3: MANEJO DE MISSING

In [ ]:
# Imputar numéricos con mediana por grupo
df['x'] = df.groupby('sexo')['x'].transform(lambda s: s.fillna(s.median()))
# Imputar categóricos con nueva categoría
df['cat'] = df['cat'].fillna('Unknown')
# Indicador
df['x_missing'] = df['x'].isna().astype(int)

## 🎨 FASE 4: FEATURE ENGINEERING

In [ ]:
# Ratio
df['ratio'] = df['a'] / (df['b']+1e-6)
# Binning
df['edad_grp'] = pd.cut(df['edad'], bins=[0,18,35,60,100], labels=['<18','18-35','35-60','60+'])
# Fechas cíclicas
df['mes'] = df['fecha'].dt.month
df['mes_sin'] = np.sin(2*np.pi*df['mes']/12)
df['mes_cos'] = np.cos(2*np.pi*df['mes']/12)
# Logaritmo
df['x_log'] = np.log1p(df['x'])

## 🏷️ FASE 5: ENCODING CATEGÓRICAS

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
X_cat = ohe.fit_transform(df[['cat']])
cat_cols = ohe.get_feature_names_out(['cat'])
X_cat = pd.DataFrame(X_cat, columns=cat_cols, index=df.index)
df = pd.concat([df, X_cat], axis=1).drop(columns=['cat'])

## ⚖️ FASE 6: OUTLIERS

In [ ]:
from scipy.stats import mstats
df['x_win'] = mstats.winsorize(df['x'], limits=[0.01,0.01])

## 🔄 FASE 7: BALANCEO (solo clasificación)

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X, y)

## 📏 FASE 8: ESCALADO

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## 🔍 FASE 9: SELECCIÓN DE CARACTERÍSTICAS

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
sel = SelectKBest(chi2, k=20)
X_new = sel.fit_transform(X_train, y_train)
selected = X_train.columns[sel.get_support()]

## ✂️ FASE 10: DIVISIÓN

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

## 🎯 FASE 11: VALIDACIÓN

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
clf = RandomForestClassifier(random_state=42, n_estimators=200)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))

## 📦 FASE 12: PIPELINE & SERIALIZACIÓN

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))])
pre = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)])
model = Pipeline([
    ('prep', pre),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))])
model.fit(X_train, y_train)

In [ ]:
# Guardar todo
import joblib, datetime, os
version = datetime.datetime.now().strftime('%Y%m%d_%H%M')
os.makedirs('artifacts', exist_ok=True)
joblib.dump(model, f'artifacts/pipeline_{version}.pkl')
print('Pipeline guardado en:', f'artifacts/pipeline_{version}.pkl')